In [19]:
from ib_insync import IB, Forex, util
import pandas as pd
import numpy as np
import nest_asyncio
import asyncio

In [20]:
# Apply the patch to allow nested event loops
nest_asyncio.apply()

In [57]:
def fetch_data(symbol, duration='6 M', bar_size='5 mins'):
    ib = IB()
    ib.connect('127.0.0.1', 7497, clientId=1, timeout=300)
    contract = Forex(symbol)
    ib.qualifyContracts(contract)
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr=duration,
        barSizeSetting=bar_size,
        whatToShow='MIDPOINT',
        useRTH=True,
        formatDate=1
    )
    df = util.df(bars)
    ib.disconnect()
    return df

In [58]:
data = fetch_data('EURUSD')


reqHistoricalData: Timeout for Forex('EURUSD', conId=12087792, exchange='IDEALPRO', localSymbol='EUR.USD', tradingClass='EUR.USD')


In [ ]:
data


In [ ]:
# Save the DataFrame to a CSV file
csv_filename = '5min-6month-EURUSD-IB.csv'
data.to_csv(csv_filename, index=True)
print(f"Historic data saved to {csv_filename}")

Historic data saved to 15min-1year-EURUSD-IB.csv


In [52]:
# Save the DataFrame to a CSV file with custom columns and order
def save_IBdata_to_csv(data, csv_filename):
    columns_order = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    columns_rename = {
    'date': 'Date',    
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
}
    # Rename columns
    data = data.rename(columns=columns_rename)
    # Reorder columns
    data = data[columns_order]
    # Save to CSV
    data.to_csv(csv_filename, index=True)  # Set index to True to include the 'date' column
    print(f"Data saved to {csv_filename} with custom columns.")



In [54]:
save_IBdata_to_csv(data, '15min-1year-EURUSD-IB2.CSV')

Data saved to 15min-1year-EURUSD-IB2.CSV with custom columns.


In [ ]:
def calculate_atr(data, period=14):
    data['H-L'] = data['high'] - data['low']
    data['H-Cp'] = abs(data['high'] - data['close'].shift(1))
    data['L-Cp'] = abs(data['low'] - data['close'].shift(1))
    data['TR'] = data[['H-L', 'H-Cp', 'L-Cp']].max(axis=1)
    data['ATR'] = data['TR'].rolling(window=period).mean()
    return data['ATR'].iloc[-1]
